# Polyp Detection — Data Preparation

Converts Kvasir-SEG segmentation masks into YOLO-seg polygon format
and creates the stratified train/val/test split.

Kvasir-SEG was originally annotated for segmentation (pixel-level masks).
This notebook extracts polygon contours from each mask using OpenCV and
normalizes them to the YOLO-seg label format, enabling instance
segmentation training without discarding boundary information.

---

## What this notebook does

1. Extracts polygon contours from Kvasir-SEG masks using cv2.findContours
2. Simplifies contours with cv2.approxPolyDP to reduce point count
3. Converts polygons to normalized YOLO-seg label format (class_id x1 y1 x2 y2 ...)
4. Splits Kvasir-SEG into train/val/test (70/15/15) using data-driven
   tertile cutoffs based on polyp area — not fixed thresholds
5. Converts CVC-ClinicDB the same way (held out as cross-dataset test set)
6. Verifies the final YOLO-seg dataset structure visually and numerically

## Split sizes

| Split | Images | Source |
|-------|--------|--------|
| Train | 700 | Kvasir-SEG |
| Val | 150 | Kvasir-SEG |
| Test | 150 | Kvasir-SEG |
| cvc_test | 612 | CVC-ClinicDB |

## Output

- data/yolo_format/train/images, train/labels
- data/yolo_format/val/images, val/labels
- data/yolo_format/test/images, test/labels
- data/yolo_format/cvc_test/images, cvc_test/labels
- configs/dataset.yaml

In [ ]:
# Import libraries

import os
import random
import shutil
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import sklearn
import yaml
from PIL import Image
from sklearn.model_selection import train_test_split

In [ ]:
# Config & Paths
# Project paths and settings

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = r"D:\Deep_Projects\polyp-detection-safety-first\repo"

PATHS = {
    "kvasir_img":  os.path.join(BASE_DIR, "data", "kvasir-seg", "Kvasir-SEG", "images"),
    "kvasir_mask": os.path.join(BASE_DIR, "data", "kvasir-seg", "Kvasir-SEG", "masks"),
    "cvc_img":     os.path.join(BASE_DIR, "data", "cvc-clinicdb", "PNG", "Original"),
    "cvc_mask":    os.path.join(BASE_DIR, "data", "cvc-clinicdb", "PNG", "Ground Truth"),

    "yolo_root":   os.path.join(BASE_DIR, "data", "yolo_format"),
    "configs":     os.path.join(BASE_DIR, "configs"),
    "figures":     os.path.join(BASE_DIR, "results", "figures"),
}

# Splits we need: train/val/test for Kvasir, plus a separate cvc_test
SPLITS = ["train", "val", "test", "cvc_test"]

for split in SPLITS:
    os.makedirs(os.path.join(PATHS["yolo_root"], split, "images"), exist_ok=True)
    os.makedirs(os.path.join(PATHS["yolo_root"], split, "labels"), exist_ok=True)

os.makedirs(PATHS["configs"], exist_ok=True)
os.makedirs(PATHS["figures"], exist_ok=True)

print("Paths configured:")
for name, path in PATHS.items():
    status = "OK" if os.path.exists(path) else "missing"
    print(f"  [{status}] {name:10s} -> {path}")

In [ ]:
# Polygon Extraction Function
# Convert a binary mask into YOLO-seg polygon format
#
# YOLO-seg label format (one line per object):
#   class_id  x1 y1  x2 y2  x3 y3  ...
# where every coordinate is normalized to [0, 1] relative to image size

def mask_to_polygons(mask, epsilon_ratio=0.002, min_area=20):
    """
    Find contours in a binary mask and simplify them into polygons.

    epsilon_ratio controls how aggressively we simplify the contour.
    A higher value means fewer points (more simplified shape).
    min_area filters out tiny noise specks that aren't real polyps.
    """
    binary = (mask > 127).astype(np.uint8) * 255

    contours, _ = cv2.findContours(
        binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    polygons = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if area < min_area:
            continue

        perimeter = cv2.arcLength(contour, True)
        epsilon   = epsilon_ratio * perimeter
        approx    = cv2.approxPolyDP(contour, epsilon, True)

        # YOLO-seg needs at least 3 points to form a polygon
        if len(approx) < 3:
            continue

        polygons.append(approx.reshape(-1, 2))

    return polygons


def normalize_polygon(polygon, img_width, img_height):
    """Scale pixel coordinates to the [0, 1] range YOLO expects."""
    normalized = polygon.astype(np.float32).copy()
    normalized[:, 0] /= img_width
    normalized[:, 1] /= img_height
    return normalized


def polygon_to_yolo_line(polygon, class_id=0):
    """Flatten a polygon into a single space-separated label line."""
    coords = polygon.flatten()
    coords_str = " ".join(f"{c:.6f}" for c in coords)
    return f"{class_id} {coords_str}"

In [ ]:
# Test Polygon Extraction on One Sample
# Quick sanity check before running this on the full dataset
# We want to see the polygon actually traces the polyp shape correctly

def load_mask(path):
    return cv2.imread(path, cv2.IMREAD_GRAYSCALE)


sample_files = sorted(os.listdir(PATHS["kvasir_mask"]))[:1]
sample_path  = os.path.join(PATHS["kvasir_mask"], sample_files[0])

mask     = load_mask(sample_path)
polygons = mask_to_polygons(mask)

print(f"Sample mask: {sample_files[0]}")
print(f"Mask size:   {mask.shape}")
print(f"Polygons found: {len(polygons)}")
for i, poly in enumerate(polygons):
    print(f"  Polygon {i}: {len(poly)} points")

# Visualize the extracted polygon on top of the original mask
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(mask, cmap="gray")
axes[0].set_title("Original Mask")
axes[0].axis("off")

axes[1].imshow(mask, cmap="gray")
for poly in polygons:
    closed = np.vstack([poly, poly[0]])
    axes[1].plot(closed[:, 0], closed[:, 1], color="lime", linewidth=2)
axes[1].set_title("Extracted Polygon")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Batch Conversion Function
# Convert an entire dataset (images + masks) into YOLO-seg format
# Copies images and writes one .txt label file per image
# Returns counts so we can verify nothing silently failed

def find_matching_mask(mask_dir, image_stem):
    for ext in (".png", ".jpg", ".jpeg", ".tif", ".tiff"):
        candidate = os.path.join(mask_dir, image_stem + ext)
        if os.path.exists(candidate):
            return candidate
    return None


def convert_dataset_to_yolo(image_files, img_dir, mask_dir, output_img_dir, output_lbl_dir):
    """
    For each image, find its mask, extract polygons, and write
    both the image copy and the YOLO label file.

    Returns counts so we can verify nothing silently failed.
    """
    converted = 0
    skipped_no_mask = 0
    skipped_no_polygon = 0

    for fname in image_files:
        stem = os.path.splitext(fname)[0]
        mask_path = find_matching_mask(mask_dir, stem)

        if mask_path is None:
            skipped_no_mask += 1
            continue

        img_path = os.path.join(img_dir, fname)
        img      = Image.open(img_path)
        w, h     = img.size

        mask     = load_mask(mask_path)
        polygons = mask_to_polygons(mask)

        if len(polygons) == 0:
            skipped_no_polygon += 1
            continue

        # Write label file (always .txt, even if source image is .jpg)
        label_lines = []
        for poly in polygons:
            norm_poly = normalize_polygon(poly, w, h)
            label_lines.append(polygon_to_yolo_line(norm_poly))

        label_path = os.path.join(output_lbl_dir, stem + ".txt")
        with open(label_path, "w") as f:
            f.write("\n".join(label_lines))

        # Copy image as-is
        dst_img_path = os.path.join(output_img_dir, fname)
        shutil.copy2(img_path, dst_img_path)

        converted += 1

    return {
        "converted": converted,
        "skipped_no_mask": skipped_no_mask,
        "skipped_no_polygon": skipped_no_polygon,
    }

In [ ]:
# Load File Lists
# Gather image filenames for both datasets before splitting

kvasir_images = sorted([
    f for f in os.listdir(PATHS["kvasir_img"])
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

cvc_images = sorted([
    f for f in os.listdir(PATHS["cvc_img"])
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

print(f"Kvasir-SEG images:   {len(kvasir_images)}")
print(f"CVC-ClinicDB images: {len(cvc_images)}")

In [ ]:
# Stratified Train/Val/Test Split
# Split Kvasir-SEG into train/val/test
#
# We stratify by polyp area using quantile-based buckets (tertiles)
# rather than fixed thresholds, so the buckets reflect the actual
# distribution of this dataset instead of an arbitrary guess.
# This matters because polyp size is one of the main difficulty
# factors in this task - without stratification, the test set
# could end up with mostly large (easy) or mostly small (hard)
# polyps by chance.

polyp_ratios_list = []
for fname in kvasir_images:
    stem = os.path.splitext(fname)[0]
    mask_path = find_matching_mask(PATHS["kvasir_mask"], stem)
    mask = load_mask(mask_path)
    ratio = (mask > 127).sum() / mask.size
    polyp_ratios_list.append(ratio)

polyp_ratios_arr = np.array(polyp_ratios_list)
q33, q66 = np.percentile(polyp_ratios_arr, [33, 66])

print(f"Polyp area tertile cutoffs: {q33*100:.1f}% / {q66*100:.1f}%")

size_buckets = []
for ratio in polyp_ratios_list:
    if ratio < q33:
        size_buckets.append("small")
    elif ratio < q66:
        size_buckets.append("medium")
    else:
        size_buckets.append("large")

print("\nPolyp size distribution (tertiles):")
for bucket in ["small", "medium", "large"]:
    count = size_buckets.count(bucket)
    print(f"  {bucket:8s}: {count} ({count/len(size_buckets)*100:.1f}%)")

# First split: 70% train, 30% temp (which becomes val + test)
train_files, temp_files, train_buckets, temp_buckets = train_test_split(
    kvasir_images, size_buckets,
    test_size=0.30,
    stratify=size_buckets,
    random_state=SEED
)

# Second split: divide the 30% temp into val (15%) and test (15%)
val_files, test_files = train_test_split(
    temp_files,
    test_size=0.50,
    stratify=temp_buckets,
    random_state=SEED
)

print(f"\nSplit sizes:")
print(f"  Train: {len(train_files)} ({len(train_files)/len(kvasir_images)*100:.1f}%)")
print(f"  Val:   {len(val_files)} ({len(val_files)/len(kvasir_images)*100:.1f}%)")
print(f"  Test:  {len(test_files)} ({len(test_files)/len(kvasir_images)*100:.1f}%)")

In [ ]:
# Run Conversion for All Splits
# Convert each split into YOLO-seg format
# CVC-ClinicDB is treated as a separate held-out test set

split_definitions = {
    "train":    train_files,
    "val":      val_files,
    "test":     test_files,
    "cvc_test": cvc_images,
}

results = {}

for split_name, file_list in split_definitions.items():
    is_cvc = split_name == "cvc_test"

    img_dir  = PATHS["cvc_img"]  if is_cvc else PATHS["kvasir_img"]
    mask_dir = PATHS["cvc_mask"] if is_cvc else PATHS["kvasir_mask"]

    out_img_dir = os.path.join(PATHS["yolo_root"], split_name, "images")
    out_lbl_dir = os.path.join(PATHS["yolo_root"], split_name, "labels")

    print(f"Converting {split_name} ({len(file_list)} images)...")

    stats = convert_dataset_to_yolo(
        file_list, img_dir, mask_dir, out_img_dir, out_lbl_dir
    )
    results[split_name] = stats

    print(f"  Converted: {stats['converted']}")
    print(f"  Skipped (no mask):     {stats['skipped_no_mask']}")
    print(f"  Skipped (no polygon):  {stats['skipped_no_polygon']}")
    print()

print("=" * 40)
print("CONVERSION SUMMARY")
print("=" * 40)
for split_name, stats in results.items():
    print(f"{split_name:10s}: {stats['converted']} images converted")

In [ ]:
# Write dataset.yaml
# Write the YOLO dataset config file
# This tells YOLOv8 where to find each split and what classes exist
#
# Note: "test" points to Kvasir-SEG's held-out split (same distribution
# as train/val). "cvc_test" is a separate key we use manually for
# cross-dataset evaluation in notebook 03 - YOLO ignores extra keys
# during training, so this is safe to include directly here.

dataset_config = {
    "path":     PATHS["yolo_root"].replace("\\", "/"),
    "train":    "train/images",
    "val":      "val/images",
    "test":     "test/images",
    "cvc_test": "cvc_test/images",
    "nc":       1,
    "names":    ["polyp"],
}

config_path = os.path.join(PATHS["configs"], "dataset.yaml")
with open(config_path, "w") as f:
    yaml.dump(dataset_config, f, default_flow_style=False, sort_keys=False)

print(f"Saved -> {config_path}")
print()
with open(config_path) as f:
    print(f.read())

print("IMPORTANT for notebook 03:")
print("  model.val(split='test')      -> evaluates on Kvasir-SEG held-out test")
print("  model.val(split='cvc_test')  -> evaluates on CVC-ClinicDB (cross-dataset)")

In [ ]:
# Verify YOLO Format Visually
# Draw converted labels back on the original images to confirm
# the polygon extraction pipeline is correct end to end

def yolo_label_to_polygons(label_path, img_width, img_height):
    polygons = []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            coords = list(map(float, parts[1:]))
            points = np.array(coords).reshape(-1, 2)
            points[:, 0] *= img_width
            points[:, 1] *= img_height
            polygons.append(points)
    return polygons


def verify_split(split_name, n=4):
    img_dir = os.path.join(PATHS["yolo_root"], split_name, "images")
    lbl_dir = os.path.join(PATHS["yolo_root"], split_name, "labels")

    files   = sorted(os.listdir(img_dir))
    samples = random.sample(files, min(n, len(files)))

    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    fig.suptitle(f"YOLO-seg Verification: {split_name}", fontweight="bold")

    for ax, fname in zip(axes, samples):
        stem = os.path.splitext(fname)[0]
        img_path = os.path.join(img_dir, fname)
        lbl_path = os.path.join(lbl_dir, stem + ".txt")

        img = np.array(Image.open(img_path).convert("RGB"))
        h, w = img.shape[:2]

        ax.imshow(img)
        if os.path.exists(lbl_path):
            polygons = yolo_label_to_polygons(lbl_path, w, h)
            for poly in polygons:
                closed = np.vstack([poly, poly[0]])
                ax.plot(closed[:, 0], closed[:, 1], color="lime", linewidth=2)
        ax.set_title(stem[:15], fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    save_path = os.path.join(PATHS["figures"], f"yolo_verify_{split_name}.png")
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved -> {save_path}")


verify_split("train")
verify_split("cvc_test")

In [ ]:
# Final Counts Summary

print("FINAL DATASET STRUCTURE")
print("=" * 40)

for split_name in SPLITS:
    img_dir = os.path.join(PATHS["yolo_root"], split_name, "images")
    lbl_dir = os.path.join(PATHS["yolo_root"], split_name, "labels")

    n_images = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    n_labels = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0

    print(f"{split_name:10s}: {n_images} images, {n_labels} labels")

print()
print("Note: cvc_test is a held-out cross-dataset test set,")
print("      not used during training - only for final evaluation.")

In [ ]:
# Summary

print("NOTEBOOK 02 COMPLETE")
print("=" * 40)
print(f"Train: {len(train_files)} images")
print(f"Val:   {len(val_files)} images")
print(f"Test:  {len(test_files)} images")
print(f"CVC cross-dataset test: {len(cvc_images)} images")
print()
print("Config saved:")
print(f"  {os.path.join(PATHS['configs'], 'dataset.yaml')}")
print()
print("Next -> 03_baseline_training.ipynb")
print("  - Train YOLOv8m-seg with default settings")
print("  - Evaluate on val set")
print("  - This becomes our baseline for comparison")